In [ ]:
# optimizatoin


In [ ]:
# optimize_all.py

import os
import glob
import pandas as pd
import numpy as np
import optuna
from optuna.importance import get_param_importances
from datetime import datetime

# ─────────────────────────────────────────────────────────────
# 경로 설정 (simulate_all.py 와 동일한 폴더 구조)
# ─────────────────────────────────────────────────────────────
PROCESSED_FOLDER  = r"C:\Users\LabPC\OneDrive\주식\Processed Data"
MACRO_FOLDER      = r"C:\Users\LabPC\OneDrive\주식\Macro Data"
PARAMETERS_FOLDER = r"C:\Users\LabPC\OneDrive\주식\Results\Parameters"
os.makedirs(PARAMETERS_FOLDER, exist_ok=True)
PARAMETERS_PATH   = os.path.join(PARAMETERS_FOLDER, "parameters.xlsx")

# ─────────────────────────────────────────────────────────────
# 1) 처리 가능한 종목 자동 추출 & 선택
# ─────────────────────────────────────────────────────────────
all_files = glob.glob(os.path.join(PROCESSED_FOLDER, "*_지표포함.csv"))
available = sorted({os.path.basename(f).split("_")[0] for f in all_files})

print("🔔 처리 가능한 종목 목록:")
for i, comp in enumerate(available, start=1):
    print(f"  {i}. {comp}")

sel = input("\n처리할 종목 번호(콤마로 구분) 또는 'all' 입력: ").strip().lower()
if sel == "all":
    TARGET_COMPANIES = available
else:
    idx = [int(x)-1 for x in sel.split(",") if x.strip().isdigit()]
    TARGET_COMPANIES = [available[i] for i in idx if 0 <= i < len(available)]

print(f"\n▶ 선택된 종목: {TARGET_COMPANIES}\n")

# ─────────────────────────────────────────────────────────────
# 2) 백테스트 기간 입력
# ─────────────────────────────────────────────────────────────
default_start = "2022-01-01"
default_end   = "2025-06-24"
start_in = input(f"백테스트 시작일 (YYYY-MM-DD, 기본 {default_start}): ").strip()
end_in   = input(f"백테스트 종료일 (YYYY-MM-DD, 기본 {default_end}): ").strip()
START_DATE = start_in if start_in else default_start
END_DATE   = end_in   if end_in   else default_end

print(f"\n▶ 테스트 기간: {START_DATE} ~ {END_DATE}\n")

new_records = []

# ─────────────────────────────────────────────────────────────
# 3) 종목별 최적화 루프
# ─────────────────────────────────────────────────────────────
for company in TARGET_COMPANIES:
    pattern = os.path.join(PROCESSED_FOLDER, f"{company}_*_지표포함.csv")
    files   = glob.glob(pattern)
    if not files:
        print(f"⚠️ 파일 없음: {pattern}")
        continue

    print(f"\n🔍 Optimizing {company} ({os.path.basename(files[0])})")

    # --- 데이터 로드 및 기간 필터 ---
    df = pd.read_csv(files[0], encoding='utf-8-sig')
    df['날짜'] = pd.to_datetime(df['날짜'])
    df = df[(df['날짜'] >= START_DATE) & (df['날짜'] <= END_DATE)].reset_index(drop=True)

    # --- VIX 병합 ---
    vix_files = glob.glob(os.path.join(MACRO_FOLDER, "*VIX*.csv"))
    if vix_files:
        vix = pd.read_csv(vix_files[0], parse_dates=[0], encoding='utf-8-sig', header=0)
        vix.columns = ['날짜','VIX']
        df = df.merge(vix, on='날짜', how='left')
    else:
        df['VIX'] = np.nan

    # ─────────────────────────────────────────────────────────
    # 신호 함수: VIX + RSI + Bollinger + MACD (SMA, 변화율 조건 제거)
    # ─────────────────────────────────────────────────────────
    def is_rule_buy(r, p):
        return (
            not np.isnan(r['VIX']) and r['VIX'] >= p['vix_buy_th'] and
            r['RSI (14일)']           < p['rsi_buy_th'] and
            r['종가']                 < r['볼린저밴드 하단'] * (1 + p['boll_buffer']) and
            r['MACD']                 > r['MACD 시그널']
        )

    def is_rule_sell(r, p):
        return (
            not np.isnan(r['VIX']) and r['VIX'] <= p['vix_sell_th'] and
            r['RSI (14일)'] > p['rsi_sell_th'] and
            r['종가']       > r['볼린저밴드 상단'] * (1 + p['boll_buffer']) and
            r['MACD']       < r['MACD 시그널']
        )

    # ─────────────────────────────────────────────────────────
    # ROI 계산 함수 (signal-only 백테스트)
    # ─────────────────────────────────────────────────────────
    def backtest_roi(p):
        cash, shares = 10_000.0, 0.0
        for _, r in df.iterrows():
            price = r['종가']
            if shares == 0 and is_rule_buy(r, p):
                shares, cash = cash/price, 0.0
            elif shares > 0 and is_rule_sell(r, p):
                cash, shares = shares*price, 0.0
        final = cash + shares * df.iloc[-1]['종가']
        return (final - 10_000.0) / 10_000.0 * 100

    # ─────────────────────────────────────────────────────────
    # 1) TPE 최적화 (n_trials=1500)
    # ─────────────────────────────────────────────────────────
    def obj_tpe(trial):
        return backtest_roi({
            'vix_buy_th':  trial.suggest_float("vix_buy_th",  0,100),
            'vix_sell_th': trial.suggest_float("vix_sell_th",0,100),
            'rsi_buy_th':  trial.suggest_float("rsi_buy_th",  0,100),
            'boll_buffer': trial.suggest_float("boll_buffer",0,0.1),
            'rsi_sell_th': trial.suggest_float("rsi_sell_th",0,100),
        })

    study_tpe = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler()
    )
    study_tpe.optimize(obj_tpe, n_trials=1500)
    best_tpe = study_tpe.best_params

    # ─────────────────────────────────────────────────────────
    # 2) 파라미터 중요도 계산
    # ─────────────────────────────────────────────────────────
    try:
        imp = get_param_importances(study_tpe)
    except RuntimeError:
        imp = {k: 0.0 for k in best_tpe.keys()}
    imp_k    = [k for k,v in imp.items() if v>=0.05]
    imp_cols = {f"importance_{k}": imp.get(k,0.0) for k in imp}

    # ─────────────────────────────────────────────────────────
    # 3) CMA-ES 최적화 (n_trials=500, vix_sell_th ≤ vix_buy_th 제약)
    # ─────────────────────────────────────────────────────────
    bounds = {
        'vix_buy_th':  (0,100), 'vix_sell_th': (0,100),
        'rsi_buy_th':  (0,100), 'boll_buffer': (0,0.1),
        'rsi_sell_th': (0,100),
    }
    narrow = {}
    for k in imp_k:
        lo,hi = bounds[k]
        bp    = best_tpe[k]
        d     = 0.2*(hi-lo)
        narrow[k] = (max(lo,bp-d), min(hi,bp+d))

    def obj_cma(trial):
        p = {}
        for k,(lo,hi) in bounds.items():
            if k in imp_k:
                lo2,hi2 = narrow[k]
                if k=="vix_sell_th":
                    hi2 = min(hi2, best_tpe["vix_buy_th"])
                p[k] = trial.suggest_float(k, lo2, hi2)
            else:
                p[k] = best_tpe[k]
        return backtest_roi(p)

    study_cma = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.CmaEsSampler()
    )
    study_cma.optimize(obj_cma, n_trials=500)
    best_cma  = study_cma.best_params
    final_roi = study_cma.best_value

    # ─────────────────────────────────────────────────────────
    # 4) 결과 기록
    # ─────────────────────────────────────────────────────────
    rec = {
        "종목":        company,
        "Start":       START_DATE,
        "End":         END_DATE,
        "ROI(%)":      round(final_roi,2),
        "OptimizedAt": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        **best_tpe, **best_cma, **imp_cols
    }
    new_records.append(rec)
    print(f"✅ Optimized: {company} → ROI {final_roi:.2f}%")

# ─────────────────────────────────────────────────────────────
# 5) parameters.xlsx 에 append & 인덱스 재부여
# ─────────────────────────────────────────────────────────────
if os.path.exists(PARAMETERS_PATH):
    existing = pd.read_excel(PARAMETERS_PATH)
    updated  = pd.concat([existing, pd.DataFrame(new_records)], ignore_index=True)
else:
    updated = pd.DataFrame(new_records)

if 'Index' in updated.columns:
    updated = updated.drop(columns=['Index'])
updated.insert(0, 'Index', range(1, len(updated)+1))
updated.to_excel(PARAMETERS_PATH, index=False)

print(f"\n🏁 Parameters updated → {PARAMETERS_PATH}")


In [ ]:
#simulation

In [ ]:
# simulate_all.py

import os
import glob
import pandas as pd
import numpy as np

# ─────────────────────────────────────────────────────────────
# 경로 설정 (코드 1 과 동일)
# ─────────────────────────────────────────────────────────────
PROCESSED_FOLDER = r"C:\Users\LabPC\OneDrive\주식\Processed Data"
MACRO_FOLDER     = r"C:\Users\LabPC\OneDrive\주식\Macro Data"
RESULTS_ROOT     = r"C:\Users\LabPC\OneDrive\주식\Results"
PARAM_FILE       = os.path.join(RESULTS_ROOT, "Parameters", "parameters.xlsx")

os.makedirs(RESULTS_ROOT, exist_ok=True)

# ─────────────────────────────────────────────────────────────
# 1) parameters.xlsx 로드 후 사용자 선택
# ─────────────────────────────────────────────────────────────
dfp = pd.read_excel(PARAM_FILE)

print("🔔 시뮬레이션 가능 인덱스 목록:")
for _, row in dfp.iterrows():
    idx, comp, s, e, roi = (
        int(row['Index']), row['종목'], row['Start'], row['End'], row['ROI(%)']
    )
    print(f"  {idx}. {comp} ({s} ~ {e}, ROI: {roi:.2f}%)")

sel = input("\n시뮬레이션할 Index 번호(콤마로 구분) 또는 'all' 입력: ").strip()
if sel.lower() == 'all':
    selected = dfp.copy()
else:
    nums = [int(x) for x in sel.split(',') if x.strip().isdigit()]
    selected = dfp[dfp['Index'].isin(nums)].copy()

# ▶️ 날짜 수동 입력
custom_start, custom_end = [], []
for _, row in selected.iterrows():
    comp = row['종목']
    print(f"\n📌 종목: {comp}")
    s = input("  시작일 입력 (예: 2020-01-01) → ").strip()
    e = input("  종료일 입력 (예: 2024-12-31) → ").strip()
    custom_start.append(pd.to_datetime(s))
    custom_end.append(pd.to_datetime(e))

selected['Start'] = custom_start
selected['End']   = custom_end

print("\n▶ 선택 및 사용자 지정 날짜:")
print(selected[['Index','종목','Start','End','ROI(%)']].to_string(index=False))
print()

# ─────────────────────────────────────────────────────────────
# 신호 함수 (VIX + RSI + Bollinger + MACD 추가, OBV 제거)
# ─────────────────────────────────────────────────────────────
def is_rule_buy(r, p):
    return (
        not np.isnan(r['VIX']) and r['VIX'] >= p['vix_buy_th'] and
        r['RSI (14일)']           < p['rsi_buy_th'] and
        # Bollinger lower band
        r['종가']                 < r['볼린저밴드 하단'] * (1 + p['boll_buffer']) and
        # MACD above signal
        r['MACD']                 > r['MACD 시그널'] and
        # simple moving average filter
        r['SMA 5일']              > r['SMA 10일'] and r['SMA 5일'] < r['SMA 60일'] and
        # 과거 가격/거래량 변화율 조건 (변경 없음)
        r['가격 상승률 (2주)']     < p['tw_price_th'] and
        r['가격 상승률 (3개월)']   < p['tm_price_th'] and
        r['거래량 상승률 (2주)']   < p['tw_vol_th'] and
        r['거래량 상승률 (3개월)'] < p['tm_vol_th']
    )

def is_rule_sell(r, p):
    return (
        not np.isnan(r['VIX']) and r['VIX'] <= p['vix_sell_th'] and
        r['RSI (14일)'] > p['rsi_sell_th'] and
        # Bollinger upper band
        r['종가']       > r['볼린저밴드 상단'] * (1 + p['boll_buffer']) and
        # MACD below signal
        r['MACD']       < r['MACD 시그널']
    )

# ─────────────────────────────────────────────────────────────
# 백테스트 함수 (code2 로직 그대로) + 빈 로그 처리
# ─────────────────────────────────────────────────────────────
def run_backtest(df, params, extra_on_buy=False, cooldown_days=0):
    cash, shares = 10_000.0, 0.0
    total_injected = 10_000.0
    logs, last_buy_date = [], None

    for _, r in df.iterrows():
        date, price = r['날짜'], r['종가']
        ok = last_buy_date is None or (date - last_buy_date).days >= cooldown_days

        # → 매수
        if is_rule_buy(r, params) and ok:
            if extra_on_buy:
                cash += 10_000.0
                total_injected += 10_000.0
            shares += cash / price
            cash = 0.0
            last_buy_date = date
            logs.append([
                date,
                f"BUY_RULE (VIX≥{params['vix_buy_th']:.2f}, RSI<{params['rsi_buy_th']:.2f}, "
                f"Price<BollLower, MACD>Signal)",
                price, shares, cash, cash + shares * price, total_injected
            ])

        # → 매도
        elif shares > 0 and is_rule_sell(r, params):
            cash += shares * price
            shares = 0.0
            last_buy_date = None
            logs.append([
                date,
                f"SELL_RULE (VIX≤{params['vix_sell_th']:.2f}, RSI>{params['rsi_sell_th']:.2f}, "
                f"Price>BollUpper, MACD<Signal)",
                price, shares, cash, cash, total_injected
            ])

    # → 최종 청산
    if shares > 0:
        date, price = df.iloc[-1]['날짜'], df.iloc[-1]['종가']
        cash += shares * price
        logs.append([date, "LIQUIDATE", price, 0.0, cash, cash, total_injected])

    cols = ["날짜","액션","가격","보유주","현금","총자산","투입금액"]
    out = pd.DataFrame(logs, columns=cols)
    if out.empty:
        return out

    out["ROI(%)"] = (
        out["총자산"].astype(float) / out["투입금액"].astype(float) * 100
    ).round(2).map(lambda x: f"{x:.2f}%")
    for c in ["현금","총자산","투입금액"]:
        out[c] = out[c].astype(float).map(lambda x: f"{x:,.0f}")
    return out

# ─────────────────────────────────────────────────────────────
# 3) 시뮬레이션 & 베이스라인 (code1 방식)
# ─────────────────────────────────────────────────────────────
for _, row in selected.iterrows():
    idx, comp = int(row['Index']), row['종목']
    start, end = row['Start'], row['End']
    params = row.drop(['Index','종목','Start','End','ROI(%)','OptimizedAt']).to_dict()

    # 지표 포함 CSV 로드 & 날짜 필터 (종료일 포함)
    fp = glob.glob(os.path.join(PROCESSED_FOLDER, f"{comp}_*_지표포함.csv"))[0]
    df_raw = pd.read_csv(fp, encoding='utf-8-sig')
    df_raw['날짜'] = pd.to_datetime(df_raw['날짜'])
    df = df_raw[(df_raw['날짜'] >= start) & (df_raw['날짜'] <= end)].reset_index(drop=True)

    # VIX 병합 (와일드카드 + NaN)
    vix_files = glob.glob(os.path.join(MACRO_FOLDER, "*VIX*.csv"))
    if not vix_files:
        df['VIX'] = np.nan
    else:
        v = pd.read_csv(vix_files[0], parse_dates=[0], encoding='utf-8-sig', header=0)
        v.columns = ['날짜','VIX']
        df = df.merge(v, on='날짜', how='left')

    out_dir = os.path.join(RESULTS_ROOT, comp)
    os.makedirs(out_dir, exist_ok=True)

    # ① 한 번만 투자
    df_once = run_backtest(df, params, extra_on_buy=False, cooldown_days=0)
    if not df_once.empty:
        roi_once = float(df_once["ROI(%)"].iloc[-1].rstrip('%'))
        fname1 = f"{idx}_{comp}_once_{start.date()}_{end.date()}_ROI_{roi_once:.2f}.csv"
        df_once.to_csv(os.path.join(out_dir, fname1), index=False, encoding='utf-8-sig')

    # ② 매수마다 추가 투자
    df_extra = run_backtest(df, params, extra_on_buy=True, cooldown_days=0)
    if not df_extra.empty:
        roi_extra = float(df_extra["ROI(%)"].iloc[-1].rstrip('%'))
        fname2 = f"{idx}_{comp}_extra_{start.date()}_{end.date()}_ROI_{roi_extra:.2f}.csv"
        df_extra.to_csv(os.path.join(out_dir, fname2), index=False, encoding='utf-8-sig')

    # ③ Baseline 1: 전체 기간 최저가 → 최고가
    min_p, max_p = df['종가'].min(), df['종가'].max()
    roi_base1 = (max_p / min_p - 1) * 100
    df_base1 = pd.DataFrame([
        [ df.loc[df['종가'].idxmin(),'날짜'], f"BUY at {min_p:.2f}", min_p, 10_000/min_p, pd.NA, 10_000, f"{roi_base1:.2f}%" ],
        [ df.loc[df['종가'].idxmax(),'날짜'], f"SELL at {max_p:.2f}", max_p, 0.0, (10_000/min_p)*max_p, (10_000/min_p)*max_p, f"{roi_base1:.2f}%" ],
    ], columns=["날짜","액션","가격","보유주","현금","총자산","ROI(%)"])
    fname3 = f"{idx}_{comp}_가장저점매수_고점매도_{start.date()}_{end.date()}_ROI_{roi_base1:.2f}.csv"
    df_base1.to_csv(os.path.join(out_dir, fname3), index=False, encoding='utf-8-sig')

    # ④ Baseline 2: 일별 DCA
    days = (end - start).days or 1
    total_inj = days * 10_000.0
    daily_amt = total_inj / days
    logs, shares = [], 0.0
    for _, r in df.iterrows():
        date, price = r['날짜'], r['종가']
        shares += daily_amt / price
        logs.append([date, "DCA_BUY", price, shares, daily_amt, total_inj])
    df_dca = pd.DataFrame(logs, columns=["날짜","액션","가격","보유주","투입금액","총투입"])
    df_dca["총자산_num"] = df_dca["보유주"] * df_dca["가격"]
    df_dca["ROI_num"]    = df_dca["총자산_num"] / df_dca["총투입"] * 100
    roi_base2 = df_dca["ROI_num"].iloc[-1]
    df_dca["총자산"]   = df_dca["총자산_num"].map(lambda x: f"{x:,.0f}")
    df_dca["ROI(%)"]   = df_dca["ROI_num"].round(2).map(lambda x: f"{x:.2f}%")
    df_dca["투입금액"] = df_dca["투입금액"].map(lambda x: f"{x:,.0f}")
    df_dca["총투입"]   = df_dca["총투입"].map(lambda x: f"{x:,.0f}")
    df_dca = df_dca[["날짜","액션","가격","보유주","투입금액","총자산","총투입","ROI(%)"]]
    fname4 = f"{idx}_{comp}_분할매수시나리오_{start.date()}_{end.date()}_ROI_{roi_base2:.2f}.csv"
    df_dca.to_csv(os.path.join(out_dir, fname4), index=False, encoding='utf-8-sig')

    print(f"✅ {comp} 시뮬레이션 완료: "
          f"{fname1 if not df_once.empty else '(no trades)'}, "
          f"{fname2 if not df_extra.empty else '(no trades)'}, "
          f"{fname3}, {fname4}")
